## API Client With Bearer Token

This notebook demonstrates the deployed authentication pattern for the API:
1. Sign in a user with Microsoft Entra ID via device code flow (no browser pop-up required from the notebook kernel).
2. Acquire a token for this API audience.
3. Call `/chat/stream` with `Authorization: Bearer <token>`.
4. The API performs server-side OBO to Fabric.

> **Required before running code cells:** The API service must already be running and reachable at `CHAT_API_BASE_URL` (default: `http://localhost:8000`).

> **Start the API from the repo root:**
```bash
uv run langgraph-fabric-api
```

> **VS Code alternative:** Run task `run-api`.
> **Quick check:** Open `http://localhost:8000/docs` in a browser. If it loads, the API is running.

> **Required environment variables:**
- `CHAT_API_BASE_URL` (default: `http://localhost:8000`)
- `API_CLIENT_ID` (API app registration client ID)
- `CALLER_CLIENT_ID` (public client app registration client ID)
- `TENANT_ID` (optional, default: `common`)
- `API_SCOPE` (optional, default: `api://<API_CLIENT_ID>/access_as_user`)

For device code flow, `CALLER_CLIENT_ID` must be a public client app registration (not the API server app registration).

> **Prerequisite:** Configure a public calling app registration with delegated permission to this API scope (`api://<API_CLIENT_ID>/access_as_user`).


In [ ]:
from __future__ import annotations

import os

import httpx
import msal

API_BASE_URL = os.getenv("CHAT_API_BASE_URL", "http://localhost:8000")
API_CLIENT_ID = os.getenv("API_CLIENT_ID", "")
CALLER_CLIENT_ID = os.getenv("CALLER_CLIENT_ID", "")
TENANT_ID = os.getenv("TENANT_ID", "common")
API_SCOPE = os.getenv("API_SCOPE", "")

In [ ]:
def acquire_api_token() -> str:
    """Acquire a token for the API using device code flow."""
    if not API_CLIENT_ID:
        raise ValueError("Set API_CLIENT_ID to your API app registration's Client ID")

    if not CALLER_CLIENT_ID:
        raise ValueError("Set CALLER_CLIENT_ID to your public calling application's Client ID")

    if API_CLIENT_ID == CALLER_CLIENT_ID:
        raise ValueError(
            "API_CLIENT_ID and CALLER_CLIENT_ID must be different app registrations. "
            "Use a separate public client app for CALLER_CLIENT_ID."
        )

    scope = API_SCOPE or f"api://{API_CLIENT_ID}/access_as_user"

    app = msal.PublicClientApplication(
        CALLER_CLIENT_ID,
        authority=f"https://login.microsoftonline.com/{TENANT_ID}",
    )

    flow = app.initiate_device_flow(scopes=[scope])
    if "user_code" not in flow:
        raise ValueError(f"Failed to create device flow: {flow}")

    print(flow.get("message", "Complete device sign-in in your browser."))
    result = app.acquire_token_by_device_flow(flow)

    if "access_token" not in result:
        error_description = result.get("error_description", result.get("error", "Unknown error"))
        if "AADSTS7000218" in error_description:
            raise ValueError(
                "Failed to acquire token (AADSTS7000218). Your CALLER_CLIENT_ID is being treated as a confidential client. "
                "Fixes: (1) use a separate public client app registration for CALLER_CLIENT_ID, "
                "(2) in that app set 'Allow public client flows' to Yes, "
                "(3) ensure delegated permission api://<API_CLIENT_ID>/access_as_user is granted."
            )
        raise ValueError(f"Failed to acquire token: {error_description}")

    return result["access_token"]


def stream_chat(api_token: str, prompt: str) -> str:
    """Send a streaming chat request to the API and collect SSE text output."""
    request_body = {"prompt": prompt}
    output_buffer = []

    with httpx.stream(
        "POST",
        f"{API_BASE_URL}/chat/stream",
        json=request_body,
        headers={
            "Accept": "text/event-stream",
            "Authorization": f"Bearer {api_token}",
        },
        timeout=120,
    ) as response:
        response.raise_for_status()

        current_event = None
        current_data = None

        for line in response.iter_lines():
            if line == "":
                continue

            if line.startswith(":"):
                continue

            if line.startswith("event:"):
                # Flush previous event if we have text data
                if current_event == "text" and current_data is not None:
                    output_buffer.append(current_data)
                elif current_event == "done":
                    return "".join(output_buffer)
                current_event = line[len("event:") :].strip()
                current_data = None
                continue

            if line.startswith("data:"):
                # SSE spec: strip only the single optional space after "data:"
                raw = line[len("data:") :]
                data = raw[1:] if raw.startswith(" ") else raw
                if current_data is None:
                    current_data = data
                else:
                    # Restore the newline that _format_sse_event split across data: lines
                    current_data += "\n" + data

        # Flush any remaining data
        if current_event == "text" and current_data:
            output_buffer.append(current_data)

        result = "".join(output_buffer)
        if not result:
            return "[No LLM output received]"
        return result

In [ ]:
token = acquire_api_token()
result = stream_chat(token, "What can you help me analyze in my Fabric data?")
print(result)